# Synthure data engine + model training (Colab, GPU)

Runs the trained data engine and retrains the Synthure models on it. Use a GPU runtime (Runtime -> Change runtime type -> GPU).

**Files this notebook calls**
- `ml/data_engine/build.py` -> trains the note generator, samples a conditional synthetic corpus, attaches independent labels, writes `ml/artifacts/corpus/{train,val,test}.jsonl` (test = frozen real-note holdout).
- `ml/train.py` -> retrains the five models on that corpus. Note type is trained in PyTorch; missing / readiness / reranker use scikit-learn. Exports to `frontend/lib/models/` and prints synthetic-val vs real-test.

**Inputs you provide once (open license, no access gates)**
- `mtsamples.csv` (real dictated notes with report-type labels).
- `medsecid.json` (optional: human section annotations, exported as a list of `{text, note_type, sections:[{name,label,start,end}]}`).

Nothing here needs a credential.

## 0. Setup: clone the branch and install deps

In [ ]:
!git clone https://github.com/aravinds-kannappan/Synthure.git
%cd Synthure
!git checkout feature/portal-fusion-and-trained-data-engine
!pip -q install torch scikit-learn numpy tokenizers
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 1. Provide the open-license corpora

Upload `mtsamples.csv` (required) and `medsecid.json` (optional). They land in the repo root `/content/Synthure/`.

In [ ]:
from google.colab import files
print('Select mtsamples.csv (and optionally medsecid.json):')
uploaded = files.upload()
print('uploaded:', list(uploaded))

## 2. Build the corpus (train the generator, sample, label)

Drop `--medsecid` if you did not upload it. `--steps` is generator training steps; `--per-type` is synthetic notes sampled per note type.

In [ ]:
%cd /content/Synthure/ml/data_engine
!python build.py --mtsamples /content/Synthure/mtsamples.csv --medsecid /content/Synthure/medsecid.json --steps 3000 --per-type 300 --device cuda

### Peek: a few generated notes and the corpus split

In [ ]:
import json
from pathlib import Path
for split in ['train', 'val', 'test']:
    rows = [json.loads(l) for l in Path(f'/content/Synthure/ml/artifacts/corpus/{split}.jsonl').read_text().splitlines()]
    src = {}
    for r in rows:
        src[r['source']] = src.get(r['source'], 0) + 1
    print(f'{split}: {len(rows)} notes  {src}')
print('\n--- sample generated notes ---')
gen = [json.loads(l) for l in Path('/content/Synthure/ml/artifacts/corpus/train.jsonl').read_text().splitlines() if json.loads(l)['source'] == 'generator'][:3]
for r in gen:
    print(f"[{r['note_type']}] {r['note'][:180]}\n")

## 3. Retrain the five models on the corpus

`train.py` trains note type in PyTorch and the rest in scikit-learn, exports to `frontend/lib/models/`, and prints synthetic-val vs the frozen real-test accuracy.

In [ ]:
%cd /content/Synthure/ml
!python train.py

## 4. Ship the updated artifacts

Download a zip, or push to the branch (uncomment and set a token).

In [ ]:
!cd /content/Synthure && zip -r /content/synthure_models.zip frontend/lib/models ml/artifacts/generator ml/artifacts/models 2>/dev/null
from google.colab import files
files.download('/content/synthure_models.zip')

In [ ]:
# Optional: push the retrained artifacts back to the branch.
# %cd /content/Synthure
# !git config user.email 'you@example.com' && git config user.name 'you'
# !git add frontend/lib/models ml/artifacts/generator && git commit -m 'retrain on data engine corpus'
# !git push https://<GITHUB_TOKEN>@github.com/aravinds-kannappan/Synthure.git feature/portal-fusion-and-trained-data-engine

**Honest framing.** The headline number is the frozen real-note test split (`test.jsonl`), reported next to the synthetic-val number, not instead of it. A note-type accuracy well below 1.00 on the real split is the expected, honest signal that the model learned clinical language rather than a template grammar.